# English deprivation: keep different measures distinct

**Legacy compatibility example.** This uses the older dataset-specific
`deprivation` interface. For general MHCLG publication retrieval, use the
[current provider workflow](https://xangma.github.io/ssdu/guides/provider-workflow.html).

This example shows how SSDU organises deprivation data so you can inspect each
measure separately. A score, rank and decile describe different things, even
when they refer to the same area. We will keep those distinctions and the
source information when saving and reopening the table.

**All values and labels are independently invented teaching material, not
published statistics.** Geography codes illustrate selection tokens only. This
example runs without downloading provider data. Start with
[Your first research dataset](https://xangma.github.io/ssdu/start/research-tutorial.html) if you are new to SSDU;
[installation](https://xangma.github.io/ssdu/start/install.html) explains how to open notebooks.

In [ ]:
import sys
from pathlib import Path

from ssdu import Request, Store

# Jupyter may start the kernel in this notebook's directory.
for checkout in (Path.cwd(), *Path.cwd().parents):
    if (checkout / "examples" / "dataset_contract.py").is_file():
        sys.path.insert(0, str(checkout))
        break
from examples.dataset_contract import check_dataset, fixture_client, no_network

store = Store(Path.cwd() / "store")

## One request and one retrieval API

Native selectors differ; inspect them through source metadata. Retrieval, caching and dataset access are shared.

In [ ]:
request = Request(
    provider="deprivation",
    dataset_id="english-indices-of-deprivation-2025",
    params={"scope": "england"},
    max_rows=100000,
)
with no_network(), fixture_client("deprivation") as client:
    dataset = store.fetch_dataset(request, client=client)
    fixture_http_attempts = client.attempts
assert fixture_http_attempts >= 0  # Zero on a rerun means the cache was reused.
assert dataset.snapshot_id is not None
print(dataset.snapshot_id)

## The standard observation table

The retained national input is reshaped into observations for each LSOA and measure. Scores, ranks, deciles and population measures remain separate variables; a rank is not a count. The release edition does not imply a common measurement period. Source geography vintages and references remain in the descriptor.

In [ ]:
observations = dataset.to_arrow("observations")
frame = dataset.to_pandas("observations")
check_dataset(dataset, "deprivation")
assert len(frame) == observations.num_rows
frame

## Check the source and meaning

A dataset description keeps its source, coverage and variables with the table.
Here we show the title and the meaning and unit of each measure. For a real
analysis, check these before interpreting the numbers.

In [ ]:
descriptor = dataset.contract
assert descriptor["quality"]["source_rows"] == 3
assert descriptor["quality"]["observation_rows"] == observations.num_rows
assert dataset.to_arrow("source").num_rows == 3
print("Title:", descriptor["title"])
for variable in descriptor["variables"]:
    print(variable["label"], "— unit:", variable.get("unit") or "unknown")

## Cache reuse and explicit offline replay

Pin the snapshot ID to reopen the same transformed bytes independently of provider updates. `offline=True` forbids acquiring missing source responses.

In [ ]:
with no_network():
    cached = store.fetch_dataset(request, offline=True)
    reopened = store.open_dataset(dataset.snapshot_id)
assert cached.snapshot_id == dataset.snapshot_id
assert reopened.to_arrow().equals(observations)
assert reopened.contract == descriptor
print({"offline_replay": True, "fixture_http_attempts": fixture_http_attempts})

To try a published dataset, first check [source coverage](https://xangma.github.io/ssdu/reference/sources.html)
and the provider's available areas, dates and measures. The [provider guide](https://xangma.github.io/ssdu/guides/dataset-examples.html)
explains the difference between teaching responses and live downloads.

To move saved data between machines, follow [transfer a collection](https://xangma.github.io/ssdu/start/quickstart.html).